<a href="https://colab.research.google.com/github/eduardozurek/smart-sensors/blob/main/notebooks/02_trabajo_en_grupo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# IA para sensores — Trabajo en grupo
### IBERSENSOR 2026 · Universidad del Norte

**Archivo:** `02_trabajo_en_grupo.ipynb`  
**Dataset:** WISDM, acelerómetro del smartphone (ARFF procesado)  
**Modalidad:** grupos de tres participantes · **Tiempo sugerido:** 35 minutos

**Objetivo:** reproducir el procedimiento del ejemplo guiado con **otro sujeto**, evaluar un clasificador Random Forest y contrastar los resultados entre grupos.

**Instrucciones:** ejecuten las celdas en orden; completen las respuestas Markdown y entreguen la tabla de resultados. Cada grupo utiliza un sujeto distinto, asignado por el instructor.

> **Alcance:** partición aleatoria estratificada de ventanas procesadas. Es una evaluación exploratoria **intra-sujeto**; no garantiza independencia temporal ni generalización a personas nuevas.

## 1. Identificación del equipo y configuración (3 min)

**Grupo:** _Escribir número_  
**Integrantes:** _Escribir nombres_  
**Sujeto asignado:** _Escribir identificador_

**Tarea 1:** modifiquen **únicamente** `SUJETO` según la asignación del instructor. Mantengan los demás parámetros para que los resultados sean comparables.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from io import StringIO
from urllib.request import urlopen
from scipy.io import arff
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report,
    ConfusionMatrixDisplay
)

SEED = 42
SUJETO = 1604  # TODO: sustituir por el sujeto asignado al grupo
SENSOR = "accel"
DISPOSITIVO = "phone"
ACTIVIDADES = ["A", "B", "C", "D", "E"]  # Cambiar a None para usar todas las actividades

print("Sujeto:", SUJETO)
print("Actividades seleccionadas:", ACTIVIDADES or "Todas")

## 2. Descargar los datos (4 min)

**Tarea 2:** ejecuten la lectura remota del archivo `data_<SUJETO>_accel_phone.arff`. Anoten cuántas filas y columnas contiene. No necesitan descargar el archivo manualmente.

In [ ]:
def leer_wisdm(sujeto, sensor="accel", dispositivo="phone"):
    base = "https://raw.githubusercontent.com/eduardozurek/ia/main/_005_clustering/WISDM/"
    nombre = f"data_{sujeto}_{sensor}_{dispositivo}.arff"
    url = base + nombre
    print("Leyendo:", url)
    with urlopen(url, timeout=30) as respuesta:
        contenido = respuesta.read().decode("utf-8-sig")
    datos, metadatos = arff.loadarff(StringIO(contenido))
    tabla = pd.DataFrame(datos)
    tabla.columns = tabla.columns.str.strip('"').str.strip()
    for col in tabla.select_dtypes(include=["object"]).columns:
        tabla[col] = tabla[col].map(
            lambda v: v.decode("utf-8") if isinstance(v, bytes) else v
        )
    return tabla, metadatos

try:
    df, metadatos = leer_wisdm(SUJETO, SENSOR, DISPOSITIVO)
except Exception as exc:
    raise RuntimeError(
        "No fue posible leer el ARFF. Verifica la conexión, la rama main, "
        "la ruta y el nombre del archivo en GitHub."
    ) from exc

print("Dimensiones:", df.shape)
display(df.head())

**Respuesta del grupo:** ¿Cuántas ventanas contiene el archivo? ¿Cuántas columnas?  
_Respuesta:_

## 3. Comprender las variables (3 min)

**Tarea 3:** comprueben que `ACTIVITY` contiene la actividad que queremos predecir y `class` identifica al sujeto. ¿Cuántas características quedan al excluir ambas columnas?

In [ ]:
print("Número total de columnas:", len(df.columns))
print("Columnas:")
print(df.columns.tolist())

if not {"ACTIVITY", "class"}.issubset(df.columns):
    raise ValueError("No se encontraron ACTIVITY y class. Revisa el esquema del ARFF.")

print("\nSujetos presentes:", df["class"].unique())
print("Actividades presentes:", sorted(df["ACTIVITY"].dropna().unique()))
print("Número de atributos predictivos:", len(df.columns) - 2)

if len(df.columns) != 93:
    print("Aviso: este archivo tiene un número de columnas distinto de las 93 esperadas.")

**Respuesta:** ¿Por qué `class` y `ACTIVITY` deben excluirse de `X`?  
_Respuesta:_

## 4. Explorar las actividades (4 min)

**Tarea 4:** observen la distribución de actividades. ¿Cuál tiene más ventanas? ¿Cuál tiene menos?

In [ ]:
conteos = df["ACTIVITY"].value_counts().sort_index()
display(conteos.rename_axis("Actividad").reset_index(name="Ventanas"))

fig, ax = plt.subplots(figsize=(10, 4))
conteos.plot(kind="bar", ax=ax)
ax.set(title=f"Distribución de actividades — sujeto {SUJETO}",
       xlabel="Código de actividad", ylabel="Número de ventanas")
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

**Respuesta:** Actividad más frecuente: ___ · Actividad menos frecuente: ___

## 5. Preparar las características (4 min)

Utilizaremos las mismas cinco actividades que el instructor: A (caminar), B (trotar), C (escaleras), D (sentado) y E (de pie). No cambien esta selección para la comparación entre equipos.

**Tarea 5:** ejecuten y verifiquen que `X` contiene 91 características numéricas y `y` contiene los códigos de actividad.

In [ ]:
trabajo = df.copy()
if ACTIVIDADES is not None:
    trabajo = trabajo[trabajo["ACTIVITY"].isin(ACTIVIDADES)].copy()

X = trabajo.drop(columns=["ACTIVITY", "class"])
y = trabajo["ACTIVITY"].astype(str)

if X.empty or y.nunique() < 2:
    raise ValueError("Se necesitan ejemplos de al menos dos actividades.")

X = X.apply(pd.to_numeric, errors="raise")
X = X.replace([np.inf, -np.inf], np.nan)

print("Ejemplos:", len(X))
print("Características:", X.shape[1])
print("Actividades:", sorted(y.unique()))
print("Valores faltantes por columna (solo columnas afectadas):")
display(X.isna().sum().loc[lambda s: s > 0].to_frame("Faltantes"))

if X.isna().any().any():
    raise ValueError(
        "Hay valores faltantes/no finitos. Deben tratarse mediante una "
        "estrategia de imputación ajustada exclusivamente con entrenamiento."
    )

## 6. Separar los datos y entrenar Random Forest (5 min)

Se utilizará 80 % para entrenamiento, 20 % para prueba, `random_state=42`, estratificación y 100 árboles. **No cambien estos valores** en la comparación principal.

> **Limitación:** los ARFF procesados no aportan en este ejercicio información suficiente para certificar la independencia temporal entre ventanas. La evaluación es exploratoria.

In [ ]:
conteos_seleccion = y.value_counts()
if conteos_seleccion.min() < 2:
    raise ValueError("Cada actividad necesita al menos dos ejemplos para estratificar.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)
print("Distribución entrenamiento:")
display(y_train.value_counts().sort_index().to_frame("Ventanas"))
print("Distribución prueba:")
display(y_test.value_counts().sort_index().to_frame("Ventanas"))

In [ ]:
modelo = RandomForestClassifier(
    n_estimators=100,
    random_state=SEED,
    n_jobs=-1
)
modelo.fit(X_train, y_train)
y_pred = modelo.predict(X_test)
print("Modelo entrenado; predicciones generadas:", len(y_pred))

## 7. Evaluar las predicciones (5 min)

**Tarea 6:** calculen Accuracy y F1-macro. Comparen el F1-score de las cinco actividades. ¿Cuál presenta mayor y menor desempeño?

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-macro: {f1_macro:.4f}")
print("\nReporte por actividad:")
print(classification_report(y_test, y_pred, zero_division=0))

reporte = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
clases = sorted(y.unique())
metricas_clase = pd.DataFrame(reporte).T.loc[clases, ["precision", "recall", "f1-score", "support"]]
display(metricas_clase)

**Resultados del grupo:**  
Accuracy: ___ · F1-macro: ___  
Mejor actividad según F1: ___ · Peor actividad según F1: ___

## 8. Interpretar la matriz de confusión (3 min)

**Tarea 7:** identifiquen una pareja de actividades que el clasificador confunda. Recuerden: filas = clase real; columnas = predicción.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, labels=clases,
    display_labels=clases, cmap="Blues", ax=ax,
    values_format="d", colorbar=False
)
ax.set_title(f"Matriz de confusión — sujeto {SUJETO}")
plt.tight_layout()
plt.show()

**Respuesta:** ¿Qué dos actividades se confunden y cuántos errores observan?  
_Respuesta:_

## 9. Importancia de características (2 min)

**Tarea 8:** identifiquen las tres características con mayor importancia según Random Forest. Estas importancias son exploratorias y no implican causalidad.

In [ ]:
importancias = pd.Series(
    modelo.feature_importances_, index=X.columns, name="Importancia"
).sort_values(ascending=False)

display(importancias.head(15).to_frame())

fig, ax = plt.subplots(figsize=(9, 6))
importancias.head(15).sort_values().plot(kind="barh", ax=ax)
ax.set(title="15 características más importantes", xlabel="Importancia (impureza)", ylabel="")
plt.tight_layout()
plt.show()

**Respuesta:** 1. ___ · 2. ___ · 3. ___

## 10. Registrar y compartir resultados (2 min)

**Tarea 9:** ejecuten la tabla resumen y compartan con el instructor: sujeto, número de ventanas, Accuracy, F1-macro, actividad con menor F1 y característica principal.

In [ ]:
resumen = pd.DataFrame([{
    "sujeto": SUJETO,
    "sensor": SENSOR,
    "dispositivo": DISPOSITIVO,
    "actividades": ",".join(clases),
    "ejemplos": len(X),
    "caracteristicas": X.shape[1],
    "ejemplos_entrenamiento": len(X_train),
    "ejemplos_prueba": len(X_test),
    "accuracy": round(accuracy, 4),
    "f1_macro": round(f1_macro, 4),
    "caracteristica_principal": importancias.index[0],
    "protocolo": "aleatorio estratificado; exploratorio intra-sujeto"
}])
display(resumen)

# Opcional: guardar resultados locales
# resumen.to_csv(f"resultados_wisdm_{SUJETO}.csv", index=False)

## 11. Discusión final del equipo

Completen estas respuestas para la puesta en común:

1. **Comparación:** ¿el Accuracy y el F1-macro cuentan la misma historia? ¿Por qué?
2. **Errores:** ¿qué actividad fue más difícil de reconocer? Apóyense en la matriz de confusión.
3. **Sujetos:** ¿por qué dos grupos pueden obtener métricas distintas si usan el mismo algoritmo y parámetros?
4. **Generalización:** ¿puede concluirse que este clasificador funcionará con una persona no vista? Justifiquen.
5. **Mejora:** propongan un experimento adicional (por ejemplo, otras características o más actividades), pero **sin alterar** los parámetros del resultado principal que compararemos entre grupos.

**Entrega:** tabla `resumen`, matriz de confusión y respuestas breves. Si disponen de tiempo, exporten `resumen` como CSV con la línea opcional de la celda anterior.

---
[Guía de clasificación](../docs/04_clasificacion_actividades.md) · [Dataset WISDM](../docs/03_dataset_wisdm.md) · [Inicio](../README.md)